<a href="https://colab.research.google.com/github/nishithy13/DS2002-Midterm/blob/main/06-midterm-walmart/2026-10-30%20%E2%80%94%20Midterm%20Progress%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Progress

**Lab — 2026-10-30 · Fall 2026**  

---

## Lab 10 — Midterm progress, week 2 of 3

One submission per team. Questions 1 through 3 should be substantially answered by now — not polished, but answered, with a number and a chart each.

Presentations are Monday. If a question is not working, this is the last checkpoint where there is time to do something about it, so report the actual state rather than the intended one.

### Q1 — Demand surge

**TODO:** your baseline window, your surge window, the average for each, and the multiple. Plus the chart. State the window boundaries explicitly — a multiple without a window is not a finding.

In [1]:
# TODO: the numbers
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def data_path(filename):
    for folder in ("data", "../data", "06-midterm-walmart/data", "DS2002-Midterm/data", "DS2002-Midterm/06-midterm-walmart/data", "."):
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return os.path.join("data", filename)

# Reusing our Lab 09 cleaning pipeline
def drop_duplicates(df):
    return df.drop_duplicates()

def drop_negative_quantity(df):
    qty = pd.to_numeric(df["quantity"], errors="coerce")
    return df.loc[~(qty < 0)]

def clean_unit_price(df):
    df = df.copy()
    stripped = df["unit_price"].astype("string").str.replace(r"[$,]", "", regex=True)
    df["unit_price"] = pd.to_numeric(stripped, errors="coerce")
    return df

def parse_timestamp(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    return df

def clean_store_id(df):
    df = df.copy()
    df["store_id"] = (df["store_id"].astype("string").str.strip().str.upper()
                      .str.replace("_", "-", regex=False)
                      .str.replace(" ", "-", regex=False)
                      .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
                      .replace("", pd.NA))
    return df

def drop_unusable(df):
    df = df.copy()
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    return df.dropna(subset=["unit_price", "quantity", "timestamp", "store_id"])

PIPELINE = [drop_duplicates, drop_negative_quantity, clean_unit_price,
            parse_timestamp, clean_store_id, drop_unusable]

def clean_data(raw):
    df = raw
    for step in PIPELINE:
        df = step(df)
    return df

original_df = pd.read_csv(data_path("walmart_transactions.csv"))
clean_df = clean_data(original_df)
clean_df["date"] = clean_df["timestamp"].dt.normalize()

# Reusing Lab 09 Strawberry Pop-Tarts SKU mapping
catalog = pd.read_csv(data_path("product_catalog.csv"))
catalog["sku"] = catalog["sku"].astype("string").str.strip().str.upper()
clean_df["sku"] = clean_df["sku"].astype("string").str.strip().str.upper()

pop_tarts_skus = ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"]
all_skus = pd.concat([
    original_df["sku"].astype("string").str.strip().str.upper(),
    catalog["sku"]
]).dropna().drop_duplicates()

sku_mapping = pd.DataFrame({"sku": sorted(all_skus)})
sku_mapping["canonical_product"] = sku_mapping["sku"]
sku_mapping.loc[sku_mapping["sku"].isin(pop_tarts_skus), "canonical_product"] = "Strawberry Pop-Tarts"

clean_df = clean_df.merge(sku_mapping, on="sku", how="left", validate="many_to_one")

# Baseline vs surge windows (Pattern Following the problem from the lecture in CLinic 3)
daily = clean_df[clean_df["canonical_product"] == "Strawberry Pop-Tarts"].groupby("date")["quantity"].sum().reset_index()
daily = daily.set_index("date").sort_index()

landfall = pd.Timestamp("2024-09-26")

for window in (3, 5, 7):
    start = landfall - pd.Timedelta(days=window)
    surge = daily.loc[(daily.index >= start) & (daily.index < landfall)]
    baseline = daily.loc[daily.index < start]

    surge_mean = surge["quantity"].mean()
    baseline_mean = baseline["quantity"].mean()
    mult = surge_mean / baseline_mean
    print(f"{window}-day window (Sept {start.day}-{landfall.day-1}): surge {surge_mean:6.1f} | baseline {baseline_mean:6.1f} | {mult:.1f}x")

# Official Q1 numbers using the 3-day window leading up to landfall
start_3d = landfall - pd.Timedelta(days=3)
surge_3d = daily.loc[(daily.index >= start_3d) & (daily.index < landfall)]["quantity"].mean()
baseline_full = daily.loc[daily.index < start_3d]["quantity"].mean()
q1_mult = round(surge_3d / baseline_full, 2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/walmart_transactions.csv'

In [ ]:
# TODO: the chart
# Studio Pattern 1 chart adapted for Strawberry Pop-Tarts
fig, ax = plt.subplots(figsize=(8, 3.5))

ax.plot(daily.index, daily["quantity"], marker="o", ms=3, color="#b4553f", label="Strawberry Pop-Tarts")
ax.axvline(landfall, ls="--", color="gray", lw=1)
ax.annotate("landfall", xy=(landfall + pd.Timedelta(hours=6), daily["quantity"].max() * 0.9),
            fontsize=9, color="gray")

ax.set_title("Storm demand builds three days out and peaks before landfall",
             loc="left", fontsize=12)
ax.set_ylabel("units sold")
ax.legend(frameon=False, fontsize=9)

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

**Q1 answer, in one sentence a manager could read:** The demand for consolidated Strawberry Pop-Tarts surged to 7.22x baseline in the 3 days leading up to Hurricane Helene's September 26 landfall. This means that stores need to pre-position extra inventory at least 72 hours ahead of a storm to maximize effectiveness and value.

### Q2 — SKU consolidation

**TODO:** how many distinct SKUs you started with, how many canonical products you ended with, your largest consolidated family, and a before-and-after comparison showing what changes. Include the assertion proving consolidation did not change unit totals.

In [ ]:
# TODO

**Q2 answer:** _..._

### Q3 — Weather correlation

**TODO:** the API call, the validated join, and the relationship you found. Include:

- the row count before and after the join, proving nothing was lost
- whether the data is live or fallback
- how many days the correlation is computed over

If your n is small, say so here rather than being asked about it on Monday.

In [ ]:
# TODO

**Q3 answer, with its caveat:** _..._

### Q4 and Q5 — where they stand

**TODO:** one honest sentence each on the remaining two questions and who is finishing them this weekend.

**Q4 (stores and geography):** _..._

**Q5 (signal or artifact):** _..._

### Self-check

In [ ]:
progress = {
    'q1_multiple': None,            # TODO: a number
    'q1_baseline_window': 'TODO',   # TODO: stated explicitly
    'q2_canonical_products': None,  # TODO: a number
    'q3_join_rows_preserved': None, # TODO: True/False
    'q3_days_in_correlation': None, # TODO: a number
    'charts_with_message_titles': None,  # TODO: how many
    'notebook_runs_clean': False,   # TODO: True after Restart and Run All
    'repo_pushed': False,           # TODO
}

for label, value in progress.items():
    filled = not (value is None or value is False
                  or (isinstance(value, str) and value.startswith('TODO')))
    print('PASS' if filled else 'FAIL', '-', label, '=', value)
print()
print('ready for presentations:', all(
    not (v is None or v is False or (isinstance(v, str) and v.startswith('TODO')))
    for v in progress.values()))

### Write-up

**a)** One finding that surprised your team, and what you checked to make sure it was real rather than a bug.

**b)** One number you reported that you are least confident in, and why.

_your answer here_